In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Rfp.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 147529,2017-02-14 15:34:34,UNKNOWN,organizational unit 65458,137.526306,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 147529,2017-02-14 15:34:43,UNKNOWN,organizational unit 65458,137.526306,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,9.0
2,request for payment 147529,2017-02-15 14:48:02,UNKNOWN,organizational unit 65458,137.526306,Request Payment,SYSTEM,UNDEFINED,83599.0
3,request for payment 147529,2017-02-20 17:32:08,UNKNOWN,organizational unit 65458,137.526306,Payment Handled,SYSTEM,UNDEFINED,441846.0
4,request for payment 147534,2017-03-02 15:55:43,UNKNOWN,organizational unit 65463,59.567024,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
5,request for payment 147534,2017-03-02 15:58:27,UNKNOWN,organizational unit 65463,59.567024,Request For Payment APPROVED by PRE_APPROVER,STAFF MEMBER,PRE_APPROVER,164.0
6,request for payment 147534,2017-03-02 16:07:38,UNKNOWN,organizational unit 65463,59.567024,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,551.0
7,request for payment 147534,2017-03-06 13:57:31,UNKNOWN,organizational unit 65463,59.567024,Request Payment,SYSTEM,UNDEFINED,337793.0
8,request for payment 147534,2017-03-13 17:31:05,UNKNOWN,organizational unit 65463,59.567024,Payment Handled,SYSTEM,UNDEFINED,617614.0
9,request for payment 147539,2017-03-06 14:40:07,UNKNOWN,organizational unit 65458,47.927757,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [3.00, 325445.40]                        57230.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [10.54, 665.70]                          74.7009    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
or

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load()

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

In [16]:
engine.branching_sets

[{'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR',
  'Request For Payment SUBMITTED by EMPLOYEE'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Rfp-cf_generated_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/200 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 176718,4,1,0,0.113479,0.060292,0.166667,0.444444,0.181818,...,0.739742,0.181818,0.113479,0.166667,0.060292,0.444444,0.000000,0.000000,0.0,0.000000
1,0,request for payment 166711,5,1,0,0.057811,0.115623,0.000000,0.333333,0.307692,...,0.698837,0.307692,0.057811,0.000000,0.115623,0.333333,0.000000,0.000000,0.0,0.000000
2,0,request for payment 184068,5,1,0,0.083384,0.000102,0.166667,0.333333,0.307692,...,0.724410,0.307692,0.083384,0.166667,0.000102,0.333333,0.000000,0.000000,0.0,0.000000
3,0,request for payment 182365,5,1,0,0.000017,0.000033,0.000000,0.333333,0.307692,...,0.641042,0.307692,0.000017,0.000000,0.000033,0.333333,0.000000,0.000000,0.0,0.000000
4,0,request for payment 168709,5,1,0,0.084002,0.001337,0.166667,0.444444,0.307692,...,0.836138,0.307692,0.084002,0.166667,0.001337,0.444444,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,16,request for payment 178121,9,1,0,0.063814,0.127629,0.000000,0.333333,0.619048,...,1.981341,0.619048,0.063814,0.000000,0.127629,0.333333,0.965146,0.733020,1.0,0.999999
131,16,request for payment 170670,9,1,0,0.035884,0.000340,0.071429,0.380952,0.714286,...,2.067205,0.714286,0.035884,0.071429,0.000340,0.380952,0.936083,0.000000,1.0,0.000000
132,16,request for payment 171283,9,1,0,0.082452,0.164904,0.000000,0.333333,0.619048,...,1.996914,0.619048,0.082452,0.000000,0.164904,0.333333,0.962081,0.846649,1.0,1.000000
133,16,request for payment 171342,9,1,0,0.144760,0.289520,0.000000,0.333333,0.619048,...,2.040171,0.619048,0.144760,0.000000,0.289520,0.333333,0.943030,0.858765,1.0,1.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/200 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 176718,4,1,0,0.242136,0.317606,0.166667,0.444444,0.181818,...,0.868399,0.181818,0.242136,0.166667,0.317606,0.444444,0.000000,0.0,0.0,0.0
1,0,request for payment 166711,5,1,0,0.057821,0.115643,0.000000,0.333333,0.307692,...,0.698847,0.307692,0.057821,0.000000,0.115643,0.333333,0.000000,0.0,0.0,0.0
2,0,request for payment 184068,5,1,0,0.223915,0.281164,0.166667,0.444444,0.307692,...,0.976052,0.307692,0.223915,0.166667,0.281164,0.444444,0.000000,0.0,0.0,0.0
3,0,request for payment 182365,5,1,0,0.010062,0.020123,0.000000,0.333333,0.307692,...,0.651087,0.307692,0.010062,0.000000,0.020123,0.333333,0.000000,0.0,0.0,0.0
4,0,request for payment 168709,5,1,0,0.174247,0.181828,0.166667,0.444444,0.307692,...,0.926384,0.307692,0.174247,0.166667,0.181828,0.444444,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,16,request for payment 178121,9,1,0,0.137112,0.202795,0.071429,0.380952,0.809524,...,2.295549,0.809524,0.137112,0.071429,0.202795,0.380952,0.967962,0.0,1.0,0.0
131,16,request for payment 170670,9,1,0,0.085649,0.099869,0.071429,0.380952,0.714286,...,2.118228,0.714286,0.085649,0.071429,0.099869,0.380952,0.937341,0.0,1.0,0.0
132,16,request for payment 171283,9,1,0,0.148846,0.154834,0.142857,0.428571,0.619048,...,2.144377,0.619048,0.148846,0.142857,0.154834,0.428571,0.947912,0.0,1.0,0.0
133,16,request for payment 171342,9,1,0,0.164720,0.258011,0.071429,0.380952,0.714286,...,2.203326,0.714286,0.164720,0.071429,0.258011,0.380952,0.943368,0.0,1.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()